# GLiNER2 Medical NER — Full Fine-Tuning

This notebook trains a **GLiNER2** model using **full fine-tuning** (all 205M parameters) on the prepared medical NER dataset.

## Full Fine-Tuning vs LoRA

| Aspect | Full Fine-Tuning (this notebook) | LoRA (`training.ipynb`) |
|--------|----------------------------------|-------------------------|
| Trainable params | 100% (~205M) | ~0.1-1% (~1-2M) |
| GPU requirement | V100 16GB / A100 40GB | T4 16GB |
| Training speed | Slower | 2-3x faster |
| Checkpoint size | ~450 MB | ~5-10 MB |
| Best for | Large datasets, max performance | Limited data, quick iteration |
| Overfitting risk | Higher | Lower (implicit regularization) |

> **GPU Recommendation**: Use **V100 16GB** (minimum) or **A100 40GB** (recommended).  
> On Colab: Runtime → Change runtime type → GPU → V100 or A100.

## Prerequisites
- Run `data_preparation.ipynb` first to generate the JSONL files
- Required files in `prepared_data/`: `train.jsonl`, `val.jsonl`, `test.jsonl`, `eval_ood.jsonl`
- `eval_ood.jsonl` is produced from the new `151-eval.json` out-of-distribution set

## Notebook Overview

| Step | Description |
|------|-------------|
| 1. Setup | Install dependencies, check GPU |
| 2. Configuration | Full fine-tuning hyperparameters |
| 3. Load Data | Load & validate prepared JSONL files |
| 4. Model Setup | Load GLiNER2 base model |
| 5. Training | Full fine-tuning (all parameters) |
| 6. Evaluation | In-Distribution + Out-of-Distribution (P/R/F1) |
| 7. Push to HuggingFace | Upload trained model to HF Hub |
| 8. Inference | Example usage of the trained model |

In [4]:
# Install dependencies
# Uncomment when running on Google Colab
#%pip install gliner2 "transformers<4.48" huggingface_hub

import json
import os
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
    if vram_gb < 15:
        print("⚠ WARNING: Full fine-tuning needs ≥16GB VRAM. Consider using training.ipynb (LoRA) instead.")
else:
    print("WARNING: No GPU detected. Training will be extremely slow.")
    print("On Colab: Runtime → Change runtime type → GPU (V100 or A100)")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Ti
VRAM: 16.6 GB


## 1. Configuration

All training hyperparameters in one place.

### Full Fine-Tuning Strategy

Unlike LoRA, full fine-tuning updates **all** model parameters. Key differences:
- **Two learning rates**: `encoder_lr` (backbone) and `task_lr` (task heads) are both active
- **Lower encoder LR**: The pre-trained encoder uses a smaller LR (5e-6) to preserve learned representations
- **Higher task LR**: Task-specific heads use a higher LR (1e-4) since they need more adaptation
- **Smaller batch size**: Full model in VRAM leaves less room for activations
- **More gradient accumulation**: Compensates for the smaller per-GPU batch

Reference: [GLiNER2 Training Tutorial — Medical NER Example](https://github.com/fastino-ai/GLiNER2/blob/main/tutorial/9-training.md)

In [8]:
# ============================================================
# CONFIGURATION — FULL FINE-TUNING
# ============================================================

# --- Paths ---
DATA_DIR = "./prepared_data"                     # Output from data_preparation.ipynb
TRAIN_FILE = os.path.join(DATA_DIR, "train.jsonl")
VAL_FILE = os.path.join(DATA_DIR, "val.jsonl")
TEST_FILE = os.path.join(DATA_DIR, "test.jsonl")
OOD_EVAL_FILE = os.path.join(DATA_DIR, "eval_ood.jsonl")
OUTPUT_DIR = "./gliner2_medical_ner_full-540-samples"         # Separate from LoRA output

# --- Base Model ---
BASE_MODEL = "fastino/gliner2-large-v1"           # 486M params

# --- Training Hyperparameters ---
# Reference: GLiNER2 Medical NER example (tutorial/9-training.md)
NUM_EPOCHS = 15
BATCH_SIZE = 1                                    # Reduced from 4 to avoid OOM — full model is 486M
GRADIENT_ACCUMULATION_STEPS = 2                   # Reduced from 8 for memory efficiency; effective batch = 8
ENCODER_LR = 5e-6                                 # Lower LR for pre-trained encoder
TASK_LR = 1e-4                                    # Higher LR for task heads
WARMUP_RATIO = 0.05                               # Gentle warmup
SCHEDULER = "cosine"                              # Cosine annealing
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

# --- Memory Optimization (critical for full fine-tuning) ---
FP16 = True                                       # Mixed precision — halves activation memory
NUM_WORKERS = 0                                   # Set to 0 for CPU + GPU compatibility; avoid multiprocessing issues

# --- Evaluation & Checkpointing ---
EVAL_STRATEGY = "epoch"                           # Evaluate + save at end of each epoch
SAVE_BEST = True                                  # Keep best checkpoint by val loss
SAVE_TOTAL_LIMIT = 3                              # Keep max 3 checkpoints (saves disk)
EARLY_STOPPING = True
EARLY_STOPPING_PATIENCE = 3
LOGGING_STEPS = 20

# --- HuggingFace Hub ---
HF_REPO_ID = "haiderAI/gliner2-MeDataset-NER-540-samples-OOD"   # Change this!
HF_PRIVATE = True

# --- Reproducibility ---
SEED = 42

# --- Entity types (must match data_preparation.ipynb) ---
ENTITY_TYPES = ["Dataset"]
ENTITY_DESCRIPTIONS = {
    "Dataset": "Names of datasets, databases, corpus, collection, cohort, benchmarks, or data collections used in scientific research"
}

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 50)
print("FULL FINE-TUNING CONFIGURATION")
print("=" * 50)
print(f"Base model:    {BASE_MODEL}")
print(f"Training:      {NUM_EPOCHS} epochs, batch={BATCH_SIZE}x{GRADIENT_ACCUMULATION_STEPS}={BATCH_SIZE*GRADIENT_ACCUMULATION_STEPS}")
print(f"Encoder LR:    {ENCODER_LR}")
print(f"Task LR:       {TASK_LR}")
print(f"Scheduler:     {SCHEDULER} (warmup {WARMUP_RATIO})")
print(f"FP16:          {FP16}")
print(f"Output:        {OUTPUT_DIR}")

FULL FINE-TUNING CONFIGURATION
Base model:    fastino/gliner2-large-v1
Training:      15 epochs, batch=1x2=2
Encoder LR:    5e-06
Task LR:       0.0001
Scheduler:     cosine (warmup 0.05)
FP16:          True
Output:        ./gliner2_medical_ner_full-540-samples


## 2. Load & Validate Data

Load the JSONL files generated by `data_preparation.ipynb` and validate them.

In [10]:
# Quick check files exist
for fpath in [TRAIN_FILE, VAL_FILE, TEST_FILE, OOD_EVAL_FILE]:
    if not os.path.exists(fpath):
        raise FileNotFoundError(
            f"File not found: {fpath}\n"
            f"Run data_preparation.ipynb first to generate the JSONL files."
        )

# Count examples
def count_jsonl(path):
    count = 0
    pos = 0
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                count += 1
                ex = json.loads(line)
                if any(len(v) > 0 for v in ex.get('output', {}).get('entities', {}).values()):
                    pos += 1
    return count, pos

for name, path in [("Train", TRAIN_FILE), ("Val", VAL_FILE), ("Test (ID)", TEST_FILE), ("OOD Eval", OOD_EVAL_FILE)]:
    total, pos = count_jsonl(path)
    print(f"{name:10s}: {total:5d} examples ({pos} positive, {total-pos} negative)")

print("\n✓ All data files present and readable.")

Train     :  5133 examples (4060 positive, 1073 negative)
Val       :   684 examples (562 positive, 122 negative)
Test (ID) :  1028 examples (823 positive, 205 negative)
OOD Eval  :  2826 examples (2131 positive, 695 negative)

✓ All data files present and readable.


In [11]:
# GLiNER2 validation (optional but recommended)
from gliner2.training.data import TrainingDataset

for label, path in [("Training", TRAIN_FILE), ("Validation", VAL_FILE)]:
    print(f"Loading and validating {label.lower()} data...")
    ds = TrainingDataset.load(path)
    try:
        ds.validate()
    except TypeError:
        pass
    if hasattr(ds, 'print_stats'):
        ds.print_stats()
    else:
        print(f"  {label}: {len(ds)} examples loaded ✓")
    print()

print("✓ Datasets validated.")

Loading and validating training data...
Loaded 5133 examples from prepared_data/train.jsonl

GLiNER2 Training Dataset Statistics
Total examples: 5133

Text lengths: min=338, max=1500, mean=1225.7

Task Distribution:
  entities_only: 5133 (100.0%)

Entity Types (7450 total mentions):
  Dataset: 7450


Loading and validating validation data...
Loaded 684 examples from prepared_data/val.jsonl

GLiNER2 Training Dataset Statistics
Total examples: 684

Text lengths: min=325, max=1500, mean=1187.0

Task Distribution:
  entities_only: 684 (100.0%)

Entity Types (1077 total mentions):
  Dataset: 1077


✓ Datasets validated.


## 3. Load GLiNER2 Model

Load the pre-trained base model. In full fine-tuning, **all parameters** will be updated during training.

In [12]:
from gliner2 import GLiNER2

print(f"Loading base model: {BASE_MODEL}")
model = GLiNER2.from_pretrained(BASE_MODEL)
print("✓ Model loaded.")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,} (100%)")

# Quick sanity check — run inference before training
test_text = "We evaluated our approach on the CIFAR-10 dataset and the ImageNet benchmark."
result = model.extract_entities(test_text, ENTITY_TYPES)
print(f"\nPre-training sanity check:")
print(f"  Input: {test_text}")
print(f"  Output: {result}")

Loading base model: fastino/gliner2-large-v1
🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-large
Counting layer     : count_lstm
Token pooling      : first
✓ Model loaded.
Total parameters:     486,444,053
Trainable parameters: 486,444,053 (100%)

Pre-training sanity check:
  Input: We evaluated our approach on the CIFAR-10 dataset and the ImageNet benchmark.
  Output: {'entities': {'Dataset': ['CIFAR-10', 'ImageNet']}}


## 4. Configure & Run Training

### Full Fine-Tuning — All Parameters

Full fine-tuning updates the entire model, including the pre-trained encoder backbone and all task-specific heads. This gives the model maximum capacity to adapt but requires:
- **Two separate learning rates**: A lower `encoder_lr` (5e-6) preserves pre-trained representations, while a higher `task_lr` (1e-4) lets task heads adapt quickly
- **FP16 mixed precision**: Essential to fit the full model + optimizer states in GPU VRAM
- **Cosine scheduler**: Smooth decay helps convergence for the larger parameter space
- **Early stopping (patience=3)**: Guards against overfitting since all params are free to move

Reference: [GLiNER2 Training Config](https://github.com/fastino-ai/GLiNER2/blob/main/tutorial/9-training.md#training-configuration)

In [13]:
from gliner2.training.trainer import GLiNER2Trainer, TrainingConfig

config = TrainingConfig(
    # Output
    output_dir=OUTPUT_DIR,
    experiment_name="MeDataset-Full-FineTuning-540-samples",  # Change this!

    # Training
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    # Learning rates — both active in full fine-tuning
    encoder_lr=ENCODER_LR,      # Pre-trained encoder (lower)
    task_lr=TASK_LR,            # Task heads (higher)

    # Scheduler
    warmup_ratio=WARMUP_RATIO,
    scheduler_type=SCHEDULER,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,

    # NO LoRA — full fine-tuning
    use_lora=False,

    # Memory optimization (critical for full model)
    fp16=FP16,
    num_workers=NUM_WORKERS,

    # Evaluation & checkpointing
    eval_strategy=EVAL_STRATEGY,
    save_best=SAVE_BEST,
    save_total_limit=SAVE_TOTAL_LIMIT,
    early_stopping=EARLY_STOPPING,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,

    # Logging
    logging_steps=LOGGING_STEPS,

    # Reproducibility
    seed=SEED,
)

print("Training configuration:")
print(f"  Mode:            FULL FINE-TUNING (no LoRA)")
print(f"  Epochs:          {config.num_epochs}")
print(f"  Batch size:      {config.batch_size} x {config.gradient_accumulation_steps} = {config.batch_size * config.gradient_accumulation_steps}")
print(f"  Encoder LR:      {config.encoder_lr}")
print(f"  Task LR:         {config.task_lr}")
print(f"  Scheduler:       {config.scheduler_type} (warmup {config.warmup_ratio})")
print(f"  Early stopping:  patience={config.early_stopping_patience}")
print(f"  FP16:            {config.fp16}")
print(f"  Save limit:      {config.save_total_limit} checkpoints")

Training configuration:
  Mode:            FULL FINE-TUNING (no LoRA)
  Epochs:          15
  Batch size:      1 x 2 = 2
  Encoder LR:      5e-06
  Task LR:         0.0001
  Scheduler:       cosine (warmup 0.05)
  Early stopping:  patience=3
  FP16:            True
  Save limit:      3 checkpoints


In [14]:
# Train!
trainer = GLiNER2Trainer(model, config)

print("Starting full fine-tuning...")
print("="*60)

results = trainer.train(
    train_data=TRAIN_FILE,
    eval_data=VAL_FILE,
)

print("="*60)
print("Training complete!")
print(f"  Best validation loss: {results.get('best_metric', 'N/A')}")
print(f"  Total steps:          {results.get('total_steps', 'N/A')}")
total_time = results.get('total_time_seconds', 0)
if total_time:
    print(f"  Training time:        {total_time/60:.1f} minutes")

2026-04-14 21:06:00 - INFO - gliner2.training.trainer - LoRA is disabled


Starting full fine-tuning...


Validating records: 100%|██████████| 5133/5133 [00:00<00:00, 171003.67record/s]
/home/shakoor/miniconda3/envs/taqi/lib/python3.11/site-packages/gliner2/training/trainer.py:900: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=self.config.fp16)
2026-04-14 21:06:00 - INFO - gliner2.training.trainer - ***** Running Training *****
2026-04-14 21:06:00 - INFO - gliner2.training.trainer -   Num examples = 5133
2026-04-14 21:06:00 - INFO - gliner2.training.trainer -   Num epochs = 15
2026-04-14 21:06:00 - INFO - gliner2.training.trainer -   Batch size = 1
2026-04-14 21:06:00 - INFO - gliner2.training.trainer -   Gradient accumulation steps = 2
2026-04-14 21:06:00 - INFO - gliner2.training.trainer -   Effective batch size = 2
2026-04-14 21:06:00 - INFO - gliner2.training.trainer -   Total optimization steps = 38490
2026-04-14 21:06:00 - INFO - gliner2.training.trainer -   Warmup step

Training:   0%|          | 0/38490 [00:00<?, ?it/s]

/home/shakoor/miniconda3/envs/taqi/lib/python3.11/site-packages/gliner2/training/trainer.py:947: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp, dtype=amp_dtype):
/home/shakoor/miniconda3/envs/taqi/lib/python3.11/site-packages/gliner2/training/trainer.py:993: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  self.scheduler.step()
2026-04-14 21:13:39 - INFO - gliner2.training.trainer - Flushed incomplete gradient accumulation cycle at end of epoch (grad_norm: 0.00)
2026-04-14 21:13:39 - INFO - gliner2.training.trainer - Applied incomplete gradie

...

Evaluating:   0%|          | 0/86 [00:00<?, ?it/s]

2026-04-14 21:21:47 - INFO - gliner2.training.trainer - 💾 Saved full checkpoint 'best' | step 5134 | epoch 2.0 | 486,444,053 params | 1866.3MB | 2.2s
2026-04-14 21:21:47 - INFO - gliner2.training.trainer - New best eval_loss: 26.1520
2026-04-14 21:21:48 - INFO - gliner2.training.trainer - 💾 Saved full checkpoint 'checkpoint-epoch-2' | step 5134 | epoch 2.0 | 486,444,053 params | 1866.3MB | 1.5s
2026-04-14 21:29:26 - INFO - gliner2.training.trainer - Flushed incomplete gradient accumulation cycle at end of epoch (grad_norm: 0.00)
2026-04-14 21:29:26 - INFO - gliner2.training.trainer - Applied incomplete gradient accumulation at end of epoch 3
2026-04-14 21:29:26 - INFO - gliner2.training.trainer - Epoch 3/15 - Loss: 0.8332
2026-04-14 21:29:26 - INFO - gliner2.training.trainer - Running evaluation...


Evaluating:   0%|          | 0/86 [00:00<?, ?it/s]

2026-04-14 21:29:39 - INFO - gliner2.training.trainer - Early stopping triggered at epoch 3
2026-04-14 21:29:41 - INFO - gliner2.training.trainer - 💾 Saved full checkpoint 'final' | step 7701 | epoch 3.0 | 486,444,053 params | 1866.3MB | 1.5s


Training complete!
  Best validation loss: 26.15196355476943
  Total steps:          7701
  Training time:        23.7 minutes


## 5. Load Best Model

Load the best checkpoint (selected by lowest validation loss during training).

In [15]:
best_model_path = os.path.join(OUTPUT_DIR, "best")

if os.path.exists(best_model_path):
    print(f"Loading best model from: {best_model_path}")
    best_model = GLiNER2.from_pretrained(best_model_path)
    print("✓ Best model loaded.")
else:
    # Fallback to final checkpoint
    final_path = os.path.join(OUTPUT_DIR, "final")
    print(f"Best model not found. Loading final model from: {final_path}")
    best_model = GLiNER2.from_pretrained(final_path)
    print("✓ Final model loaded.")

# Quick sanity check
test_text = "We evaluated our approach on the CIFAR-10 dataset and the ImageNet benchmark."
result = best_model.extract_entities(test_text, ENTITY_TYPES)
print(f"\nPost-training sanity check:")
print(f"  Input: {test_text}")
print(f"  Output: {result}")

Loading best model from: ./gliner2_medical_ner_full-540-samples/best
🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-large
Counting layer     : count_lstm
Token pooling      : first
✓ Best model loaded.

Post-training sanity check:
  Input: We evaluated our approach on the CIFAR-10 dataset and the ImageNet benchmark.
  Output: {'entities': {'Dataset': ['CIFAR-10 dataset', 'ImageNet benchmark']}}


## 6. In-Distribution & Out-of-Distribution Evaluation

We evaluate the trained model on **two separate test sets**:

1. **In-Distribution (ID) Test Set** — 15% held-out split from the same data source. This measures how well the model generalizes to unseen chunks from the same distribution. *Not used during training or checkpoint selection.*

2. **Out-of-Distribution (OOD) Test Set** — the new `151-eval.json` set, exported to `eval_ood.jsonl`. This measures how well the model generalizes to a separate distribution.

### Metrics
- **Exact Match** — predicted entity text must exactly match the ground truth
- **Partial Match** — predicted entity text overlaps with ground truth (substring)
- **Precision** — fraction of predictions that are correct
- **Recall** — fraction of ground truth entities that were found
- **F1** — harmonic mean of precision and recall

In [16]:
def load_eval_data(filepath):
    """Load JSONL evaluation data."""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data


def evaluate_model(model, eval_data, entity_types):
    """
    Evaluate the model on evaluation data.
    Returns per-type and overall metrics.
    """
    all_true_entities = []   # list of sets of (entity_type, entity_text)
    all_pred_entities = []   # list of sets of (entity_type, entity_text)

    for example in eval_data:
        text = example["input"]
        gold_entities = example["output"]["entities"]

        # Ground truth
        true_set = set()
        for etype, mentions in gold_entities.items():
            for mention in mentions:
                true_set.add((etype, mention.strip().lower()))

        # Predictions
        pred_result = model.extract_entities(text, entity_types)
        pred_set = set()
        pred_entities = pred_result.get("entities", {})
        for etype, mentions in pred_entities.items():
            if isinstance(mentions, list):
                for mention in mentions:
                    if isinstance(mention, dict):
                        mention = mention.get("text", "")
                    pred_set.add((etype, str(mention).strip().lower()))

        all_true_entities.append(true_set)
        all_pred_entities.append(pred_set)

    # Compute metrics
    tp = fp = fn = 0
    partial_tp = 0

    for true_set, pred_set in zip(all_true_entities, all_pred_entities):
        # Exact match
        tp += len(true_set & pred_set)
        fp += len(pred_set - true_set)
        fn += len(true_set - pred_set)

        # Partial match: check if any predicted mention is a substring of a true mention or vice versa
        for ptype, pmention in pred_set:
            if (ptype, pmention) not in true_set:
                for ttype, tmention in true_set:
                    if ptype == ttype and (pmention in tmention or tmention in pmention):
                        partial_tp += 1
                        break

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)

    partial_precision = (tp + partial_tp) / max(tp + partial_tp + fp - partial_tp, 1)
    partial_recall = (tp + partial_tp) / max(tp + partial_tp + fn - partial_tp, 1)
    partial_f1 = 2 * partial_precision * partial_recall / max(partial_precision + partial_recall, 1e-8)

    metrics = {
        "exact_match": {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn},
        "partial_match": {"precision": partial_precision, "recall": partial_recall, "f1": partial_f1, "partial_tp": partial_tp},
        "total_true": tp + fn,
        "total_pred": tp + fp,
    }
    return metrics

print("Evaluation functions defined.")

Evaluation functions defined.


In [17]:
def print_metrics(metrics, label):
    """Pretty-print evaluation metrics."""
    print("=" * 60)
    print(f"{label} EVALUATION RESULTS")
    print("=" * 60)
    print(f"\nTotal ground truth entities: {metrics['total_true']}")
    print(f"Total predicted entities:    {metrics['total_pred']}")

    print(f"\n--- Exact Match ---")
    em = metrics['exact_match']
    print(f"  Precision: {em['precision']:.4f}")
    print(f"  Recall:    {em['recall']:.4f}")
    print(f"  F1:        {em['f1']:.4f}")
    print(f"  (TP={em['tp']}, FP={em['fp']}, FN={em['fn']})")

    print(f"\n--- Partial Match ---")
    pm = metrics['partial_match']
    print(f"  Precision: {pm['precision']:.4f}")
    print(f"  Recall:    {pm['recall']:.4f}")
    print(f"  F1:        {pm['f1']:.4f}")


# --- In-Distribution Test Set ---
print("Evaluating on in-distribution test set...")
id_data = load_eval_data(TEST_FILE)
id_metrics = evaluate_model(best_model, id_data, ENTITY_TYPES)
print_metrics(id_metrics, "IN-DISTRIBUTION (ID)")

print("\n")

# --- Out-of-Distribution Test Set ---
print("Evaluating on out-of-distribution test set...")
ood_data = load_eval_data(OOD_EVAL_FILE)
ood_metrics = evaluate_model(best_model, ood_data, ENTITY_TYPES)
print_metrics(ood_metrics, "OUT-OF-DISTRIBUTION (OOD)")

# --- Side-by-side comparison ---
print("\n" + "=" * 60)
print("COMPARISON: ID vs OOD")
print("=" * 60)
print(f"{'Metric':<20} {'ID':>10} {'OOD':>10}")
print("-" * 40)
print(f"{'Exact P':<20} {id_metrics['exact_match']['precision']:>10.4f} {ood_metrics['exact_match']['precision']:>10.4f}")
print(f"{'Exact R':<20} {id_metrics['exact_match']['recall']:>10.4f} {ood_metrics['exact_match']['recall']:>10.4f}")
print(f"{'Exact F1':<20} {id_metrics['exact_match']['f1']:>10.4f} {ood_metrics['exact_match']['f1']:>10.4f}")
print(f"{'Partial P':<20} {id_metrics['partial_match']['precision']:>10.4f} {ood_metrics['partial_match']['precision']:>10.4f}")
print(f"{'Partial R':<20} {id_metrics['partial_match']['recall']:>10.4f} {ood_metrics['partial_match']['recall']:>10.4f}")
print(f"{'Partial F1':<20} {id_metrics['partial_match']['f1']:>10.4f} {ood_metrics['partial_match']['f1']:>10.4f}")

# Save all metrics
all_metrics = {"in_distribution": id_metrics, "out_of_distribution": ood_metrics}
metrics_path = os.path.join(OUTPUT_DIR, "eval_metrics.json")
with open(metrics_path, 'w') as f:
    json.dump(all_metrics, f, indent=2)
print(f"\nAll metrics saved to {metrics_path}")

Evaluating on in-distribution test set...
IN-DISTRIBUTION (ID) EVALUATION RESULTS

Total ground truth entities: 1496
Total predicted entities:    1514

--- Exact Match ---
  Precision: 0.7880
  Recall:    0.7975
  F1:        0.7927
  (TP=1193, FP=321, FN=303)

--- Partial Match ---
  Precision: 0.8554
  Recall:    0.8656
  F1:        0.8605


Evaluating on out-of-distribution test set...
OUT-OF-DISTRIBUTION (OOD) EVALUATION RESULTS

Total ground truth entities: 4212
Total predicted entities:    4337

--- Exact Match ---
  Precision: 0.7293
  Recall:    0.7509
  F1:        0.7400
  (TP=3163, FP=1174, FN=1049)

--- Partial Match ---
  Precision: 0.8162
  Recall:    0.8405
  F1:        0.8282

COMPARISON: ID vs OOD
Metric                       ID        OOD
----------------------------------------
Exact P                  0.7880     0.7293
Exact R                  0.7975     0.7509
Exact F1                 0.7927     0.7400
Partial P                0.8554     0.8162
Partial R             

In [18]:
# Qualitative examples — show predictions vs ground truth
def show_examples(data, label, model, entity_types, n=3):
    print(f"\n{'=' * 60}")
    print(f"QUALITATIVE EXAMPLES — {label} (first {n} with entities)")
    print(f"{'=' * 60}")
    shown = 0
    for ex in data:
        gold = ex["output"]["entities"]
        has_gold = any(len(v) > 0 for v in gold.values())
        if not has_gold:
            continue
        text = ex["input"]
        pred = model.extract_entities(text, entity_types)
        print(f"\n--- Example {shown+1} ---")
        print(f"Text (first 300 chars): {text[:300]}...")
        print(f"Ground truth: {gold}")
        print(f"Predicted:    {pred.get('entities', {})}")
        shown += 1
        if shown >= n:
            break

show_examples(id_data, "IN-DISTRIBUTION", best_model, ENTITY_TYPES, n=3)
show_examples(ood_data, "OUT-OF-DISTRIBUTION", best_model, ENTITY_TYPES, n=3)


QUALITATIVE EXAMPLES — IN-DISTRIBUTION (first 3 with entities)

--- Example 1 ---
Text (first 300 chars): Section 2.2. This is followed by Sections 2.3 and 2.4, which detail our two training stages that use conditional probability and a numerically stable unconditional probability formulation, respectively. Datasets and Taxonomy The first step in creating an HMLC system is to create the label taxonomy. ...
Ground truth: {'Dataset': ['PLCO dataset']}
Predicted:    {'Dataset': ['PLCO dataset']}

--- Example 2 ---
Text (first 300 chars): n reproducibility in multiple datasets (the "overlap analysis") is also applicable to other network reconstruction strategies. It has been stated by leaders in artificial intelligence and data mining that "invariably, simple models and a lot of data trump more elaborate models based on less data" . ...
Ground truth: {'Dataset': ['HIPPIE']}
Predicted:    {'Dataset': ['AML datasets']}

--- Example 3 ---
Text (first 300 chars): capes experiments in this stu

## 7. Push to HuggingFace Hub

Upload the full fine-tuned model to a HuggingFace repository.

> **Note**: Full fine-tuned checkpoints are ~450 MB (vs ~5-10 MB for LoRA adapters).

### Authentication
You need a HuggingFace token with **write access**:
1. Go to https://huggingface.co/settings/tokens
2. Create a token with "Write" permissions
3. Set it below or use `huggingface-cli login`

In [5]:
from huggingface_hub import HfApi, login

# Option 1: Login interactively (recommended on Colab)
# This will prompt you for your HF token
login()

# Option 2: Login with token directly (uncomment and set your token)
# login(token="hf_YOUR_TOKEN_HERE")

print("✓ Authenticated with HuggingFace Hub.")

✓ Authenticated with HuggingFace Hub.


In [9]:
# Upload model to HuggingFace Hub
api = HfApi()

# Determine which directory to upload (best checkpoint)
model_dir = os.path.join(OUTPUT_DIR, "best")
if not os.path.exists(model_dir):
    model_dir = os.path.join(OUTPUT_DIR, "final")

print(f"Uploading model from: {model_dir}")
print(f"Target repo: {HF_REPO_ID}")
print(f"Private: {HF_PRIVATE}")

# Create repo if it doesn't exist
api.create_repo(
    repo_id=HF_REPO_ID,
    private=HF_PRIVATE,
    exist_ok=True,
)

# Upload model files
api.upload_folder(
    folder_path=model_dir,
    repo_id=HF_REPO_ID,
    commit_message="Upload GLiNER2 medical NER model (full fine-tuned)",
)

# Also upload eval metrics
metrics_file = os.path.join(OUTPUT_DIR, "eval_metrics.json")
if os.path.exists(metrics_file):
    api.upload_file(
        path_or_fileobj=metrics_file,
        path_in_repo="eval_metrics.json",
        repo_id=HF_REPO_ID,
        commit_message="Upload evaluation metrics (ID + OOD; includes 151-eval.json)",
    )

print(f"\n✓ Model uploaded to: https://huggingface.co/{HF_REPO_ID}")

Uploading model from: ./gliner2_medical_ner_full-540-samples/best
Target repo: haiderAI/gliner2-MeDataset-NER-540-samples-OOD
Private: True


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.



✓ Model uploaded to: https://huggingface.co/haiderAI/gliner2-MeDataset-NER-540-samples-OOD


In [13]:
# Create and upload a model card (README.md) with training details

model_card = f"""---
tags:
  - gliner2
  - ner
  - medical
  - full-fine-tuning
  - entity-extraction
license: apache-2.0
---

# GLiNER2 Medical NER — Dataset Entity Extraction (Full Fine-Tuned)

Full fine-tuned [{BASE_MODEL}](https://huggingface.co/{BASE_MODEL}) model for extracting **dataset names** from medical/scientific research papers.

## Entity Types

| Entity Type | Description |
|-------------|-------------|
| `Dataset` | {ENTITY_DESCRIPTIONS.get('Dataset', 'Dataset names')} |

## Training Details

- **Base model**: `{BASE_MODEL}`
- **Method**: Full fine-tuning (all parameters)
- **Epochs**: {NUM_EPOCHS}
- **Batch size**: {BATCH_SIZE} x {GRADIENT_ACCUMULATION_STEPS} (effective {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS})
- **Encoder LR**: {ENCODER_LR}
- **Task LR**: {TASK_LR}
- **Scheduler**: {SCHEDULER} (warmup {WARMUP_RATIO})
- **FP16**: {FP16}

## Evaluation

Evaluated on two separate test sets:
- **In-Distribution (ID)**: 15% held-out split from same data source (not used during training)
- **Out-of-Distribution (OOD)**: the new `151-eval.json` set, exported to `eval_ood.jsonl`

See `eval_metrics.json` for detailed P/R/F1 metrics.

## Usage

```python
from gliner2 import GLiNER2

model = GLiNER2.from_pretrained("{HF_REPO_ID}")

text = "We evaluated our method on the CIFAR-10 and ImageNet datasets."
result = model.extract_entities(text, ["Dataset"] )
print(result)
```
"""

# Write model card to disk and upload it as repo root README.md
readme_path = os.path.join(OUTPUT_DIR, "README.md")
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(model_card)

api.upload_file(
    path_or_fileobj=readme_path,
    path_in_repo="README.md",
    repo_id=HF_REPO_ID,
    commit_message="Upload model card (README.md)",
)

print(f"✓ Model card uploaded to: https://huggingface.co/{HF_REPO_ID}")

✓ Model card uploaded to: https://huggingface.co/haiderAI/gliner2-MeDataset-NER-540-samples-OOD


## 8. Inference Examples

Demonstrate how to use the full fine-tuned model for inference on new texts.

In [14]:
# Example usage of the trained model

test_texts = [
    "We used the MIMIC-III database and PhysioNet data for patient outcome prediction.",
    "The model was benchmarked on CIFAR-10, CIFAR-100, and the full ImageNet dataset.",
    "Our ECG classification approach was tested on the MIT-BIH Arrhythmia Database and the PTB Diagnostic ECG Database.",
    "Feature extraction was performed using standard NLP techniques without any specific dataset.",
]

print("=" * 60)
print("INFERENCE EXAMPLES")
print("=" * 60)

for i, text in enumerate(test_texts):
    # Basic extraction
    result = best_model.extract_entities(text, ENTITY_TYPES)

    # With descriptions for better accuracy
    result_desc = best_model.extract_entities(text, ENTITY_DESCRIPTIONS)

    print(f"\n--- Example {i+1} ---")
    print(f"  Text: {text}")
    print(f"  Entities (basic):        {result.get('entities', {})}")
    print(f"  Entities (w/ desc):      {result_desc.get('entities', {})}")

print("\n" + "=" * 60)
print("Done! Full fine-tuned model is ready for deployment.")
print(f"Load from HF: GLiNER2.from_pretrained('{HF_REPO_ID}')")

INFERENCE EXAMPLES


NameError: name 'best_model' is not defined